<a href="https://colab.research.google.com/github/Slipstream45/random/blob/master/Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from re import X
import math
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self._children = set(_children)
    self._op = _op
    self.label = label
    self.grad = 0.0
    self._backward = lambda: None
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    output = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0*output.grad
      other.grad += 1.0*output.grad
    output._backward = _backward
    return output
  def __mul__(self, other):
    output = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data*output.grad
      other.grad += self.data*output.grad
    output._backward = _backward
    return output
  def __neg__(self):
    output = Value(self*-1.0, (self,), '-1')

    def _backward():
      self.grad -= 1*output.grad
    output._backward = _backward
    return output
  def __sub__(self, other):
    output = Value(self.data - other.data, (self, other), '-')

    def _backward():
      self.grad += 1.0*output.grad
      other.grad -= 1.0*output.grad
    output._backward = _backward
    return output
  def __pow__(self, other):
    output = Value(self.data ** other.data, (self, other), '^')

    def _backward():
      self.grad += other.data*(self.data**(other.data-1))*output.grad
    output._backward = _backward
    return output
  def tanh(self):
    x = self.data
    e = math.e
    t = (1-e**(-2*x))/(1+e**(-2*x))
    output = Value(t, (self,), 'tanh')

    def _backward():
      t = 1 - output.data**2
      self.grad += t*output.grad
    output._backward = _backward
    return output
  def exp(self):
    x = self.data
    e = math.e
    t = e**x
    output = Value(t, (self,), 'e^x')

    def _backward():
      self.grad += t*output.grad

    output._backward = _backward
    return output

  def backprop(self):
    topo=[]
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._children:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0

    for node in reversed(topo):
      node._backward()




In [ ]:
'''
a = Value(0.005)
a.label = 'a'
b = Value(1.0)
b.label = 'b'
c = a+b
c.label = 'c'
d = Value(2.0)
d.label = 'd'
e = c*d
e.label = 'e'
f = Value(2.0)
f.label = 'f'
g = e**f
g.label = 'g'
h = g.tanh()
h.label = 'h'
i = h.exp()
i.label = 'i'
print(f"FORWARD PASS DONE: i is {i.data:.4f}")

i.backprop()
'''
a = Value(0.00001, label = 'a')
b = Value(1.0, label = 'b')
d = Value(2.0, label = 'd')
f = Value(2.0, label = 'f')
learn_rate = 0.2
steps = 440
for step in range(steps):
  c = a + b
  e = c * d
  g = e ** f
  h = g.tanh()
  i = h.exp()
  n = Value(2.0)
  y = Value(1.0)
  diff = (i-y)**n

  for v in [a,b,c,d,e,f,g,h,i,diff]:
    v.grad = 0.0
  diff.backprop()

  a.data -= learn_rate * a.grad
  b.data -= learn_rate * b.grad
  print(f"PASS {step} | Loss: {diff.data:.6f} | i: {i.data:.4f} | a.data: {a.data:.6f}")
print('*'*30)
'''
print("-" * 30)
print(f"{'Node':<10} | {'Data':<10} | {'Gradient':<10}")
print("-" * 30)
print(f"{'i (exp)':<10} | {i.data:<10.4f} | {i.grad:<10.4f}")
print(f"{'h (tanh)':<10} | {h.data:<10.4f} | {h.grad:<10.4f}")
print(f"{'g (pow)':<10} | {g.data:<10.4f} | {g.grad:<10.4f}")
print(f"{'e (mul)':<10} | {e.data:<10.4f} | {e.grad:<10.4f}")
print(f"{'d (const)':<10} | {d.data:<10.4f} | {d.grad:<10.4f}")
print(f"{'c (add)':<10} | {c.data:<10.4f} | {c.grad:<10.4f}")
print(f"{'a (input)':<10} | {a.data:<10.4f} | {a.grad:<10.4f}")
print(f"{'b (input)':<10} | {b.data:<10.4f} | {b.grad:<10.4f}")
print("-" * 30)
print(f"Sensitivity of 'i' to 'a': {a.grad:.8f}")
'''

PASS 0 | Loss: 2.946233 | i: 2.7165 | a.data: -0.019995
PASS 1 | Loss: 2.940784 | i: 2.7149 | a.data: -0.055883
PASS 2 | Loss: 2.918798 | i: 2.7084 | a.data: -0.151033
PASS 3 | Loss: 2.599197 | i: 2.6122 | a.data: -0.884929
PASS 4 | Loss: 2.794455 | i: 2.6717 | a.data: -0.507551
PASS 5 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507507
PASS 6 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507464
PASS 7 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507421
PASS 8 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507379
PASS 9 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507338
PASS 10 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507297
PASS 11 | Loss: 0.000001 | i: 1.0009 | a.data: -0.507257
PASS 12 | Loss: 0.000001 | i: 1.0008 | a.data: -0.507218
PASS 13 | Loss: 0.000001 | i: 1.0008 | a.data: -0.507179
PASS 14 | Loss: 0.000001 | i: 1.0008 | a.data: -0.507141
PASS 15 | Loss: 0.000001 | i: 1.0008 | a.data: -0.507104
PASS 16 | Loss: 0.000001 | i: 1.0008 | a.data: -0.507067
PASS 17 | Loss: 0.000001 | i: 1.0008 | a.

'\nprint("-" * 30)\nprint(f"{\'Node\':<10} | {\'Data\':<10} | {\'Gradient\':<10}")\nprint("-" * 30)\nprint(f"{\'i (exp)\':<10} | {i.data:<10.4f} | {i.grad:<10.4f}")\nprint(f"{\'h (tanh)\':<10} | {h.data:<10.4f} | {h.grad:<10.4f}")\nprint(f"{\'g (pow)\':<10} | {g.data:<10.4f} | {g.grad:<10.4f}")\nprint(f"{\'e (mul)\':<10} | {e.data:<10.4f} | {e.grad:<10.4f}")\nprint(f"{\'d (const)\':<10} | {d.data:<10.4f} | {d.grad:<10.4f}")\nprint(f"{\'c (add)\':<10} | {c.data:<10.4f} | {c.grad:<10.4f}")\nprint(f"{\'a (input)\':<10} | {a.data:<10.4f} | {a.grad:<10.4f}")\nprint(f"{\'b (input)\':<10} | {b.data:<10.4f} | {b.grad:<10.4f}")\nprint("-" * 30)\nprint(f"Sensitivity of \'i\' to \'a\': {a.grad:.8f}")\n'